In [ ]:
import getpass
import json
import os
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/lora-activation-steering")
REVISION = "c591007514d5617f156a2635ead07c9bf2d0861e"


def run_command(*args):
    args = [str(arg) for arg in args]
    process = subprocess.Popen(
        args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
    except BaseException:
        process.terminate()
        process.wait()
        raise
    finally:
        process.stdout.close()
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, args)


In [ ]:
if not REPO.exists():
    run_command(
        "git", "clone", "--branch", "work",
        "https://github.com/danilov-e/lora-activation-steering.git", REPO,
    )

run_command("git", "-C", REPO, "fetch", "origin", REVISION)
run_command("git", "-C", REPO, "checkout", "--detach", REVISION)
os.chdir(REPO)
%pip install -e ".[test]"


In [ ]:
run_command(sys.executable, "-m", "pytest", "-q")


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
RUNS = Path("/content/drive/MyDrive/persona_vectors_runs")
ANSWERS = RUNS / "extraction_01" / "answers.jsonl"
JUDGE_DIR = RUNS / "judge_01"
SELECTION_DIR = RUNS / "selection_01"

assert ANSWERS.is_file(), str(ANSWERS)


In [ ]:
from dotenv import load_dotenv

load_dotenv(REPO / ".env", override=False)
if os.environ.get("OPENAI_API_KEY") in (None, "", "YOUR_APINET_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("APINet API key: ")
if not os.environ["OPENAI_API_KEY"].strip():
    raise ValueError("OPENAI_API_KEY is empty.")
os.environ.setdefault("OPENAI_BASE_URL", "https://apinet.cloud/v1")


In [ ]:
judge_command = [
    sys.executable, "-m", "scripts.judge_answers",
    "--answers", ANSWERS, "--output", JUDGE_DIR,
]
smoke_command = [*judge_command, "--limit", "2"]
if (JUDGE_DIR / "manifest.json").exists():
    smoke_command.append("--resume")

run_command(*smoke_command)


In [ ]:
summary = json.loads((JUDGE_DIR / "summary.json").read_text(encoding="utf-8"))
print(json.dumps(summary, indent=2))

with (JUDGE_DIR / "judgments.jsonl").open(encoding="utf-8") as stream:
    for i, line in enumerate(stream):
        if i == 4:
            break
        row = json.loads(line)
        print(row["id"], row["metric"], row["score"], row["numeric_mass"])


In [ ]:
run_command(*judge_command, "--resume")

summary = json.loads((JUDGE_DIR / "summary.json").read_text(encoding="utf-8"))
assert summary["status"] == "complete"
assert summary["completed_judgments"] == summary["expected_judgments"]
print(json.dumps(summary, indent=2))


In [ ]:
run_command(
    sys.executable, "-m", "scripts.filter_pairs",
    "--answers", ANSWERS, "--judgments", JUDGE_DIR,
    "--output", SELECTION_DIR,
)


In [ ]:
summary = json.loads((SELECTION_DIR / "summary.json").read_text(encoding="utf-8"))
print(json.dumps(summary, indent=2))
